In [0]:
%sql
--Crear electrocasa_dev
CREATE CATALOG IF NOT EXISTS electrocasa_dev;

In [0]:
%sql
--Crear bronze / silver / gold
CREATE SCHEMA IF NOT EXISTS electrocasa_dev.bronze;
CREATE SCHEMA IF NOT EXISTS electrocasa_dev.silver;
CREATE SCHEMA IF NOT EXISTS electrocasa_dev.gold;

In [0]:
%sql
--Crear External Volume
CREATE EXTERNAL VOLUME IF NOT EXISTS electrocasa_dev.bronze.landing
LOCATION 'abfss://electrocasa@saassociatejm01.dfs.core.windows.net/landing';

In [0]:
#Crear carpetas automatizado y no usando la interfaz 

ruta = "/Volumes/electrocasa_dev/bronze/landing"

carpetas = [
    "ventas",
    "productos",
    "empleados",
    "resenas",
    "devoluciones",
    "checkpoints",
    "schemas"
]

for carpeta in carpetas:
    dbutils.fs.mkdirs(f"{ruta}/{carpeta}")

display(dbutils.fs.ls(ruta))

In [0]:
%sql
--GRANTS
GRANT USE CATALOG
ON CATALOG electrocasa_dev
TO `electrocasa_engineers`;

GRANT USE CATALOG
ON CATALOG electrocasa_dev
TO `electrocasa_analysts`;

GRANT USE CATALOG
ON CATALOG electrocasa_dev
TO `electrocasa_auditors`;

In [0]:
%sql

--*Ingeniería puede trabajar en las tres capas
GRANT USE SCHEMA
ON SCHEMA electrocasa_dev.bronze
TO `electrocasa_engineers`;

GRANT USE SCHEMA
ON SCHEMA electrocasa_dev.silver
TO `electrocasa_engineers`;

GRANT USE SCHEMA
ON SCHEMA electrocasa_dev.gold
TO `electrocasa_engineers`;

--*Analistas solo acceden a Gold
GRANT USE SCHEMA
ON SCHEMA electrocasa_dev.gold
TO `electrocasa_analysts`;

--*Auditoría solo accede a Gold
GRANT USE SCHEMA
ON SCHEMA electrocasa_dev.gold
TO `electrocasa_auditors`;

In [0]:
%sql

-- Ingeniería: lectura y modificación en las tres capas
GRANT SELECT, MODIFY
ON SCHEMA electrocasa_dev.bronze
TO `electrocasa_engineers`;

GRANT SELECT, MODIFY
ON SCHEMA electrocasa_dev.silver
TO `electrocasa_engineers`;

GRANT SELECT, MODIFY
ON SCHEMA electrocasa_dev.gold
TO `electrocasa_engineers`;


-- Analistas: solo lectura en Gold
GRANT SELECT
ON SCHEMA electrocasa_dev.gold
TO `electrocasa_analysts`;


-- Auditoría: solo lectura en Gold
GRANT SELECT
ON SCHEMA electrocasa_dev.gold
TO `electrocasa_auditors`;

In [0]:
%sql

SHOW GRANTS ON SCHEMA electrocasa_dev.gold;

In [0]:
%sql
--REVOKES
-- Analistas no deben acceder a Bronze ni Silver
REVOKE USE SCHEMA
ON SCHEMA electrocasa_dev.bronze
FROM `electrocasa_analysts`;

REVOKE USE SCHEMA
ON SCHEMA electrocasa_dev.silver
FROM `electrocasa_analysts`;

REVOKE SELECT
ON SCHEMA electrocasa_dev.bronze
FROM `electrocasa_analysts`;

REVOKE SELECT
ON SCHEMA electrocasa_dev.silver
FROM `electrocasa_analysts`;


-- Auditoría tampoco debe acceder a Bronze ni Silver
REVOKE USE SCHEMA
ON SCHEMA electrocasa_dev.bronze
FROM `electrocasa_auditors`;

REVOKE USE SCHEMA
ON SCHEMA electrocasa_dev.silver
FROM `electrocasa_auditors`;

REVOKE SELECT
ON SCHEMA electrocasa_dev.bronze
FROM `electrocasa_auditors`;

REVOKE SELECT
ON SCHEMA electrocasa_dev.silver
FROM `electrocasa_auditors`;

In [0]:
%sql
--validacion de grants
SHOW GRANTS ON SCHEMA electrocasa_dev.bronze;

In [0]:
#Validacion de los secrets creados, sus valores estan el portal de azure key vaul.t
usuario = dbutils.secrets.get(
    scope="electrocasa-secrets",
    key="sql-user"
)

password = dbutils.secrets.get(
    scope="electrocasa-secrets",
    key="sql-password"
)

assert usuario
assert password

print("Secretos de Azure SQL disponibles correctamente")
#print(usuario)
#print(password)

In [0]:
%sql
--CREACION DE CONEXIÓN
CREATE CONNECTION IF NOT EXISTS electrocasa_sql
TYPE SQLSERVER
OPTIONS (
  host 'analyticsdmc.database.windows.net',
  port '1433',
  user secret('electrocasa-secrets', 'sql-user'),
  password secret('electrocasa-secrets', 'sql-password')
);

In [0]:
%sql
--CREACION DE CATALOGOS FOREIGN
CREATE FOREIGN CATALOG IF NOT EXISTS electrocasa_sql_catalog
USING CONNECTION electrocasa_sql
OPTIONS (
  database 'electrocasadb'
);

## Produccion

In [0]:
%sql

CREATE CATALOG IF NOT EXISTS electrocasa_prod;

CREATE SCHEMA IF NOT EXISTS electrocasa_prod.bronze;
CREATE SCHEMA IF NOT EXISTS electrocasa_prod.silver;
CREATE SCHEMA IF NOT EXISTS electrocasa_prod.gold;

CREATE EXTERNAL VOLUME IF NOT EXISTS electrocasa_prod.bronze.landing
LOCATION 'abfss://electrocasa@saassociatejm01.dfs.core.windows.net/prod/landing';

In [0]:
ruta = "/Volumes/electrocasa_prod/bronze/landing"

carpetas = [
    "ventas",
    "productos",
    "empleados",
    "resenas",
    "devoluciones",
    "schemas/ventas",
    "schemas/empleados",
    "schemas/resenas",
    "schemas/devoluciones"
]

for carpeta in carpetas:
    dbutils.fs.mkdirs(f"{ruta}/{carpeta}")

print("Carpetas de produccion creadas correctamente")